In [1]:
# =============================================================================
# Autor:   Thiago Souza (@engthiago1979-blip)
# Criado:  19-04-2026  |  Revisado: 06-06-2026 (v3)
# Skill:   analista-python-ambiental
# Uso:     Estatística avançada + 4 camadas de valor sobre a v2.1:
#            (1) Pós-hoc de Dunn  (scikit-posthocs) apos Kruskal-Wallis;
#            (2) Detecção de ruptura/regime-shift (ruptures);
#            (3) Validação de schema/tipos/domínios (pandera);
#            (4) Outliers por IQR.
#          Mantém: P95, Mann-Kendall, plots storytelling PT-BR, paleta NESA.
# Nota:    VMPs marcados como "conferir na base" — NAO validados automaticamente
#          nesta execução (Project Knowledge indisponível no Claude Code CLI).
# =============================================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats
import pymannkendall as mk
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore", message="invalid value encountered")
warnings.filterwarnings("ignore", message="Precision loss occurred")
warnings.filterwarnings("ignore", module="pandera")
_ciw = getattr(stats, 'ConstantInputWarning', None)
if _ciw is not None:
    warnings.filterwarnings("ignore", category=_ciw)

# --- Imports defensivos das camadas novas (degradam com elegância) ---
try:
    import scikit_posthocs as sp
    _HAS_DUNN = True
except Exception:
    _HAS_DUNN = False
try:
    import ruptures as rpt
    _HAS_RUPT = True
except Exception:
    _HAS_RUPT = False
try:
    import pandera.pandas as pa
    _HAS_PANDERA = True
except Exception:
    try:
        import pandera as pa
        _HAS_PANDERA = True
    except Exception:
        _HAS_PANDERA = False

# --- 1. CONFIGURAÇÕES ---
RAIZ = Path(r"C:\Users\thiago\Documents\estudos_python\etes_nesa")
DIR_BASE = RAIZ / "base_dados"
DIR_PRODUTOS = RAIZ / "Efluentes_Sanitários" / "produtos"
FILE_INPUT = DIR_BASE / "ETEs_NorteEnergia_Processado_v1.xlsx"
DIR_PRODUTOS.mkdir(parents=True, exist_ok=True)

NESA_COLORS = {
    'primary': '#009CA7', 'secondary': '#004851', 'vmp': '#F25A3C',
    'ok': '#52BD8B', 'text': '#404041', 'muted': '#9aa0a6', 'highlight': '#B9F8FF',
}

MESES_PT = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

def formatar_meses_pt(ax):
    def _fmt(x, pos=None):
        d = mdates.num2date(x)
        return f"{MESES_PT[d.month - 1]}/{d:%y}"
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(_fmt))

def configurar_estilo():
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Segoe UI', 'Calibri', 'Arial', 'DejaVu Sans'],
        'font.size': 11, 'axes.labelsize': 11, 'axes.labelcolor': NESA_COLORS['text'],
        'axes.edgecolor': '#d6d6d6', 'axes.linewidth': 1.0, 'axes.axisbelow': True,
        'axes.grid': True, 'grid.color': '#ececec', 'grid.linewidth': 0.9,
        'xtick.color': NESA_COLORS['text'], 'ytick.color': NESA_COLORS['text'],
        'text.color': NESA_COLORS['text'], 'figure.dpi': 110,
        'savefig.dpi': 300, 'savefig.bbox': 'tight', 'legend.frameon': False,
    })

configurar_estilo()

def titulo_storytelling(ax, titulo, subtitulo=None, fonte=None):
    ax.set_title('')
    ax.text(0.0, 1.13, titulo, transform=ax.transAxes, fontsize=15,
            fontweight='bold', color='#2b2b2b', va='bottom', ha='left')
    if subtitulo:
        ax.text(0.0, 1.04, subtitulo, transform=ax.transAxes, fontsize=10.5,
                color=NESA_COLORS['muted'], va='bottom', ha='left')
    if fonte:
        ax.text(0.0, -0.30, fonte, transform=ax.transAxes, fontsize=8,
                color=NESA_COLORS['muted'], va='top', ha='left')

def limpar_eixos(ax, manter_esquerda=True):
    for lado in ['top', 'right']:
        ax.spines[lado].set_visible(False)
    if not manter_esquerda:
        ax.spines['left'].set_visible(False)
    ax.grid(axis='x', visible=False)
    ax.tick_params(length=0)

# --- 2. VMPs CONAMA 430/11 (Art. 16) - CONFERIR NA BASE DO PROJETO ---
VMP_CONAMA = {
    'DEMANDA_BIOQUIMICA_DE_OXIGENIO': 120.0, 'PH': (5.0, 9.0),
    'TEMPERATURA_DO_EFLUENTE': 40.0, 'MATERIAIS_SEDIMENTAVEIS': 1.0,
    'OLEOS_E_GRAXAS': 50.0, 'ARSENIO_TOTAL': 0.5, 'CADMIO_TOTAL': 0.2,
    'CHUMBO_TOTAL': 0.5, 'CIANETO_TOTAL': 1.0, 'CROMO_HEXAVALENTE': 0.1,
    'CROMO_TRIVALENTE': 1.0, 'FERRO_DISSOLVIDO': 15.0, 'MANGANES_DISSOLVIDO': 1.0,
    'MERCURIO_TOTAL': 0.01, 'SULFETO_TOTAL': 1.0, 'BENZENO': 1.2,
    'CLOROFORMIO': 1.0, 'ETILBENZENO': 1.4, 'FENOIS_TOTAIS': 0.5,
    'TETRACLORETO_DE_CARBONO': 1.0, 'TRICLOROETENO': 1.0, 'TOLUENO': 1.2,
    'XILENO_TOTAL': 1.6,
}
UNIDADES = {'PH': '', 'TEMPERATURA_DO_EFLUENTE': '°C'}
NOTA_VMP = "VMP CONAMA 430/11 (Art. 16) - conferir na base do projeto; nao validado automaticamente nesta execucao"

# Nomes de exibição acentuados (títulos/relatórios); fallback p/ Title Case
NOMES_EXIBICAO = {
    'DEMANDA_BIOQUIMICA_DE_OXIGENIO': 'Demanda Bioquímica de Oxigênio', 'PH': 'pH',
    'TEMPERATURA_DO_EFLUENTE': 'Temperatura do Efluente', 'MATERIAIS_SEDIMENTAVEIS': 'Materiais Sedimentáveis',
    'OLEOS_E_GRAXAS': 'Óleos e Graxas', 'ARSENIO_TOTAL': 'Arsênio Total', 'CADMIO_TOTAL': 'Cádmio Total',
    'CHUMBO_TOTAL': 'Chumbo Total', 'CIANETO_TOTAL': 'Cianeto Total', 'CROMO_HEXAVALENTE': 'Cromo Hexavalente',
    'CROMO_TRIVALENTE': 'Cromo Trivalente', 'FERRO_DISSOLVIDO': 'Ferro Dissolvido',
    'MANGANES_DISSOLVIDO': 'Manganês Dissolvido', 'MERCURIO_TOTAL': 'Mercúrio Total', 'SULFETO_TOTAL': 'Sulfeto Total',
    'BENZENO': 'Benzeno', 'CLOROFORMIO': 'Clorofórmio', 'ETILBENZENO': 'Etilbenzeno', 'FENOIS_TOTAIS': 'Fenóis Totais',
    'TETRACLORETO_DE_CARBONO': 'Tetracloreto de Carbono', 'TRICLOROETENO': 'Tricloroeteno', 'TOLUENO': 'Tolueno',
    'XILENO_TOTAL': 'Xileno Total',
}

def nome_exibicao(param):
    return NOMES_EXIBICAO.get(param, param.replace('_', ' ').title())

def fmt_num(x):
    """Formata número evitando notação científica em faixas legíveis."""
    if not np.isfinite(x):
        return "-"
    a = abs(x)
    if a == 0:
        return "0"
    if a >= 100:
        return f"{x:,.0f}"
    if a >= 1:
        return f"{x:.1f}"
    if a >= 0.01:
        return f"{x:.3f}"
    return f"{x:.2g}"

# --- 3. CAMADAS NOVAS ---
def validar_schema(df, col_ete, parametros):
    """(pandera) Valida tipos/domínios: ETE textual, DATA temporal, parâmetros
    numéricos >= 0. Retorna (ok, df_relatorio). Nunca interrompe o pipeline."""
    if not _HAS_PANDERA:
        return None, pd.DataFrame([{'Verificacao': 'pandera ausente', 'Resultado': 'PULADO'}])
    cols = {col_ete: pa.Column(str, nullable=False)}
    if 'DATA' in df.columns:
        cols['DATA'] = pa.Column('datetime64[ns]', nullable=True, coerce=True)
    for p in parametros:
        cols[p] = pa.Column(float, nullable=True, coerce=True,
                            checks=pa.Check.ge(-1e-9, error="valor negativo"))
    schema = pa.DataFrameSchema(cols, strict=False)
    try:
        schema.validate(df, lazy=True)
        return True, pd.DataFrame([{'Verificacao': 'Schema (tipos+dominios)',
                                    'Resultado': 'CONFORME',
                                    'Detalhe': f'{len(parametros)} parametros, {len(df)} linhas'}])
    except pa.errors.SchemaErrors as err:
        fc = err.failure_cases[['column', 'check', 'failure_case']].copy()
        fc = fc.rename(columns={'column': 'Coluna', 'check': 'Regra', 'failure_case': 'Valor'})
        fc.insert(0, 'Resultado', 'NAO CONFORME')
        return False, fc

def contar_outliers_iqr(serie):
    """Outliers por IQR (Tukey 1.5x) - robusto p/ dados assimétricos."""
    q1, q3 = np.percentile(serie, [25, 75])
    iqr = q3 - q1
    if iqr == 0:
        return 0
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((serie < lo) | (serie > hi)).sum())

def detectar_rupturas(datas, valores, min_size=3):
    """(ruptures/PELT-l2) Detecta mudanças de regime na série. Retorna
    (n_rupturas, [datas]). Trabalha sobre o sinal padronizado (z-score)."""
    if not _HAS_RUPT:
        return 0, []
    vals = np.asarray(valores, dtype=float)
    n = len(vals)
    if n < 2 * min_size or np.nanstd(vals) == 0:
        return 0, []
    sinal = (vals - np.nanmean(vals)) / (np.nanstd(vals) + 1e-9)
    try:
        algo = rpt.Pelt(model='l2', min_size=min_size, jump=1).fit(sinal)
        bkps = algo.predict(pen=2 * np.log(n))
        idxs = [b for b in bkps if 0 < b < n]
        datas_rup = [pd.Timestamp(datas.iloc[i]).strftime('%m/%Y') for i in idxs]
        return len(idxs), datas_rup
    except Exception:
        return 0, []

def posthoc_dunn_param(df, parametro, col_ete):
    """(scikit-posthocs) Dunn pareado (ajuste de Holm) - diz QUAIS ETEs diferem."""
    if not _HAS_DUNN:
        return []
    sub = df[[col_ete, parametro]].dropna()
    validas = [e for e in sub[col_ete].unique()
               if len(sub[sub[col_ete] == e]) >= 3 and sub[sub[col_ete] == e][parametro].std() > 0]
    sub = sub[sub[col_ete].isin(validas)]
    if sub[col_ete].nunique() < 2:
        return []
    try:
        m = sp.posthoc_dunn(sub, val_col=parametro, group_col=col_ete, p_adjust='holm')
    except Exception:
        return []
    etes = list(m.columns)
    res = []
    for i in range(len(etes)):
        for j in range(i + 1, len(etes)):
            p = float(m.loc[etes[i], etes[j]])
            res.append({
                'Parâmetro': nome_exibicao(parametro),
                'ETE A': etes[i], 'ETE B': etes[j],
                'p-value (Dunn/Holm)': round(p, 4),
                'Diferem?': 'SIM' if p < 0.05 else 'NÃO',
            })
    return res

# --- 4. MOTOR ESTATÍSTICO ---
def analisar_parametro(df, parametro, limite_conama, col_ete, rupturas_detalhe):
    resultados = []
    for ete in df[col_ete].dropna().unique():
        df_ete = df[df[col_ete] == ete].sort_values('DATA')
        serie = df_ete[parametro].dropna()
        if len(serie) < 3:
            continue
        media, desvio = serie.mean(), serie.std()
        minimo, maximo = serie.min(), serie.max()
        p95 = np.percentile(serie, 95)

        if isinstance(limite_conama, tuple):
            vmin, vmax = limite_conama
            amostras_ok = len(serie[(serie >= vmin) & (serie <= vmax)])
            status_auditoria = "ROBUSTO" if (serie.min() >= vmin and p95 <= vmax) else "RISCO"
            limite_str = f"{vmin} a {vmax}"
        else:
            amostras_ok = len(serie[serie <= limite_conama])
            status_auditoria = "ROBUSTO" if p95 <= limite_conama else "RISCO"
            limite_str = str(limite_conama)
        taxa_conformidade = (amostras_ok / len(serie)) * 100

        # Mann-Kendall
        if desvio > 0 and len(serie) >= 4:
            try:
                mk_res = mk.original_test(serie)
                tendencia, p_val_mk = mk_res.trend, mk_res.p
                if tendencia == 'increasing': tendencia = 'Crescente (Piora)'
                elif tendencia == 'decreasing': tendencia = 'Decrescente (Melhora)'
                else: tendencia = 'Estável'
            except Exception:
                tendencia, p_val_mk = "Erro Cálculo", np.nan
        else:
            tendencia, p_val_mk = "Sem variância/dados", np.nan

        # NOVO: outliers (IQR) e rupturas (ruptures)
        n_out = contar_outliers_iqr(serie)
        datas_serie = df_ete.loc[serie.index, 'DATA'].reset_index(drop=True)
        n_rup, datas_rup = detectar_rupturas(datas_serie, serie.values)
        if n_rup > 0:
            rupturas_detalhe.append({
                'Parâmetro': nome_exibicao(parametro), 'ETE': ete,
                'Rupturas (n)': n_rup, 'Datas das Rupturas': ", ".join(datas_rup),
            })

        resultados.append({
            'Parâmetro': nome_exibicao(parametro), 'ETE': ete, 'N': len(serie),
            'Mínimo': round(minimo, 3), 'Média': round(media, 3),
            'Máximo': round(maximo, 3), 'Desvio Padrão': round(desvio, 3),
            'Outliers (IQR)': n_out,
            'P95 (Auditoria)': round(p95, 3),
            'VMP CONAMA (conferir base)': limite_str,
            'Conformidade (%)': f"{taxa_conformidade:.1f}%",
            'Avaliação': status_auditoria,
            'Tendência (MK)': tendencia,
            'p-value (MK)': round(p_val_mk, 4) if pd.notna(p_val_mk) else "N/A",
            'Rupturas (n)': n_rup,
        })
    return resultados

def teste_kruskal_wallis_geral(df, parametro, col_ete):
    grupos, nomes_etes = [], []
    for ete in df[col_ete].dropna().unique():
        serie = df[df[col_ete] == ete][parametro].dropna()
        if len(serie) >= 3 and serie.std() > 0:
            grupos.append(serie)
            nomes_etes.append(ete)
    if len(grupos) > 1:
        stat, p_val = stats.kruskal(*grupos)
        return {
            'Parâmetro': nome_exibicao(parametro),
            'ETEs Comparadas': " vs ".join(nomes_etes),
            'Estatística (H)': round(stat, 3), 'p-value (KW)': round(p_val, 4),
            'Diferença Significativa?': "SIM" if p_val < 0.05 else "NÃO",
        }, (p_val < 0.05)
    return None, False

def plotar_serie_temporal(df, parametro, ete, col_ete, dir_out):
    df_plot = df[df[col_ete] == ete].sort_values('DATA').dropna(subset=[parametro])
    if len(df_plot) < 2:
        return
    serie = df_plot[parametro]
    limite = VMP_CONAMA[parametro]
    p95 = np.percentile(serie, 95)
    unidade = UNIDADES.get(parametro, 'mg/L')

    fig, ax = plt.subplots(figsize=(11, 5.2))
    limpar_eixos(ax)
    ytrans = ax.get_yaxis_transform()

    if isinstance(limite, tuple):
        ax.axhspan(limite[0], limite[1], color=NESA_COLORS['ok'], alpha=0.06, zorder=0)
        for y in limite:
            ax.axhline(y, color=NESA_COLORS['vmp'], linestyle='--', linewidth=1.4, zorder=1)
            ax.text(1.005, y, f' VMP {y:g}*', transform=ytrans, color=NESA_COLORS['vmp'],
                    fontsize=9, va='center', fontweight='bold')
        topo, base = limite[1], limite[0]
    else:
        ax.axhline(limite, color=NESA_COLORS['vmp'], linestyle='--', linewidth=1.6, zorder=1)
        ax.text(1.005, limite, f' VMP {limite:g}*', transform=ytrans, color=NESA_COLORS['vmp'],
                fontsize=9, va='center', fontweight='bold')
        topo, base = limite, 0.0

    ax.plot(df_plot['DATA'], serie, marker='o', markersize=5, linewidth=2.4,
            color=NESA_COLORS['primary'], zorder=3, markerfacecolor='white',
            markeredgecolor=NESA_COLORS['primary'], markeredgewidth=1.6)

    ax.axhline(p95, color=NESA_COLORS['secondary'], linestyle=':', linewidth=1.8, zorder=2)
    ax.text(1.005, p95, f' P95 {fmt_num(p95)}', transform=ytrans, color=NESA_COLORS['secondary'],
            fontsize=9, va='center')

    # NOVO: marca pontos de ruptura de regime (ruptures)
    datas_serie = df_plot['DATA'].reset_index(drop=True)
    n_rup, datas_rup = detectar_rupturas(datas_serie, serie.values)
    for k, ts in enumerate(datas_rup):
        x = pd.to_datetime(ts, format='%m/%Y')
        ax.axvline(x, color=NESA_COLORS['muted'], linestyle='--', linewidth=1.1, alpha=0.8, zorder=1)
        if k == 0:
            ax.text(x, 1.005, ' ruptura', transform=ax.get_xaxis_transform(),
                    color=NESA_COLORS['muted'], fontsize=8, va='bottom', ha='left', rotation=0)

    x_ult, y_ult = df_plot['DATA'].iloc[-1], serie.iloc[-1]
    ax.annotate(f'{fmt_num(y_ult)}', xy=(x_ult, y_ult), xytext=(8, 0), textcoords='offset points',
                va='center', fontsize=9.5, fontweight='bold', color=NESA_COLORS['primary'])

    y_max = max(topo, float(serie.max())) * 1.12
    y_min = min(float(serie.min()), base) * 0.95 if isinstance(limite, tuple) else min(0.0, float(serie.min()))
    ax.set_ylim(y_min, y_max)

    conforme = (serie.between(limite[0], limite[1]).mean() if isinstance(limite, tuple)
                else (serie <= limite).mean()) * 100
    rotulo_y = "pH" if unidade == '' else f"Valor ({unidade})"
    extra_rup = f"  |  {n_rup} ruptura(s) de regime" if n_rup else ""
    sub = (f"{conforme:.0f}% das {len(serie)} amostras dentro do limite  |  "
           f"P95 = {fmt_num(p95)} {unidade}{extra_rup}").strip()
    titulo_storytelling(
        ax, f"{nome_exibicao(parametro)}  -  {ete}", sub,
        "Fonte: Monitoramento NESA   |   * VMP CONAMA 430/11 - conferir na base do projeto (nao validado nesta execucao)",
    )
    ax.set_ylabel(rotulo_y)
    formatar_meses_pt(ax)
    plt.setp(ax.get_xticklabels(), rotation=0, ha='center')

    fig.savefig(dir_out / f"Plot_{parametro}_{ete.replace(' ', '_')}.png")
    plt.close(fig)

# --- 5. EXECUÇÃO ---
if __name__ == "__main__":
    try:
        print(f"[*] Camadas ativas -> Dunn: {_HAS_DUNN} | Ruptures: {_HAS_RUPT} | Pandera: {_HAS_PANDERA}")
        df = pd.read_excel(FILE_INPUT)
        col_ete = [c for c in df.columns if 'DESCRICAO' in c.upper() and 'PONTO' in c.upper()][0]
        df[col_ete] = df[col_ete].astype(str).str.upper().str.strip()
        df[col_ete] = df[col_ete].replace({'ETE 1': 'ETE 01', 'ETE 2': 'ETE 02'})

        parametros_presentes = [p for p in VMP_CONAMA if p in df.columns]

        # (3) Validação de schema (pandera) - informativa
        ok_schema, df_validacao = validar_schema(df, col_ete, parametros_presentes)
        print(f"[*] Validação de schema (pandera): {'CONFORME' if ok_schema else ('NAO CONFORME' if ok_schema is False else 'PULADO')}")

        relatorio_geral, relatorio_kw, relatorio_dunn, rupturas_detalhe = [], [], [], []
        print("[*] Iniciando processamento estatístico completo (v3)...")
        for param in parametros_presentes:
            print(f"    -> Analisando: {param}")
            relatorio_geral.extend(analisar_parametro(df, param, VMP_CONAMA[param], col_ete, rupturas_detalhe))
            res_kw, significativo = teste_kruskal_wallis_geral(df, param, col_ete)
            if res_kw:
                relatorio_kw.append(res_kw)
                if significativo:  # (1) Dunn só faz sentido apos KW significativo
                    relatorio_dunn.extend(posthoc_dunn_param(df, param, col_ete))
            for ete in df[col_ete].unique():
                plotar_serie_temporal(df, param, ete, col_ete, DIR_PRODUTOS)

        # Exportação multi-aba
        caminho_excel = DIR_PRODUTOS / "Relatorio_Analitico_Mestre_v3.xlsx"
        with pd.ExcelWriter(caminho_excel, engine='openpyxl') as writer:
            pd.DataFrame(relatorio_geral).to_excel(writer, sheet_name="Estatistica_e_Tendencia", index=False)
            if relatorio_kw:
                pd.DataFrame(relatorio_kw).to_excel(writer, sheet_name="Kruskal_Wallis", index=False)
            if relatorio_dunn:
                pd.DataFrame(relatorio_dunn).to_excel(writer, sheet_name="PostHoc_Dunn", index=False)
            if rupturas_detalhe:
                pd.DataFrame(rupturas_detalhe).to_excel(writer, sheet_name="Pontos_de_Ruptura", index=False)
            df_validacao.to_excel(writer, sheet_name="Validacao_Schema", index=False)

        print(f"\n[+] SUCESSO! Relatório Mestre v3 salvo em: {caminho_excel}")
        print(f"    Abas: Estatistica_e_Tendencia | Kruskal_Wallis | PostHoc_Dunn "
              f"({len(relatorio_dunn)} pares) | Pontos_de_Ruptura ({len(rupturas_detalhe)}) | Validacao_Schema")
        print("[!] VMPs marcados como 'conferir na base do projeto' (nao validados automaticamente).")
    except Exception as e:
        print(f"[!] Erro crítico: {e}")


[*] Camadas ativas -> Dunn: True | Ruptures: True | Pandera: True
[*] Validação de schema (pandera): CONFORME
[*] Iniciando processamento estatístico completo (v3)...
    -> Analisando: DEMANDA_BIOQUIMICA_DE_OXIGENIO
    -> Analisando: PH
    -> Analisando: TEMPERATURA_DO_EFLUENTE
    -> Analisando: MATERIAIS_SEDIMENTAVEIS
    -> Analisando: OLEOS_E_GRAXAS
    -> Analisando: ARSENIO_TOTAL
    -> Analisando: CADMIO_TOTAL
    -> Analisando: CHUMBO_TOTAL
    -> Analisando: CIANETO_TOTAL
    -> Analisando: CROMO_HEXAVALENTE
    -> Analisando: CROMO_TRIVALENTE
    -> Analisando: FERRO_DISSOLVIDO
    -> Analisando: MANGANES_DISSOLVIDO
    -> Analisando: MERCURIO_TOTAL
    -> Analisando: SULFETO_TOTAL
    -> Analisando: BENZENO
    -> Analisando: CLOROFORMIO
    -> Analisando: ETILBENZENO
    -> Analisando: FENOIS_TOTAIS
    -> Analisando: TETRACLORETO_DE_CARBONO
    -> Analisando: TRICLOROETENO
    -> Analisando: TOLUENO
    -> Analisando: XILENO_TOTAL

[+] SUCESSO! Relatório Mestre v3 salvo

In [2]:
# =============================================================================
# Autor:   Thiago Souza (@engthiago1979-blip)  |  v3 (06-06-2026)
# Uso:     Módulo QA/QC: tabela + mapa de calor dos dados censurados (<LQ).
# =============================================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", message="invalid value encountered")

RAIZ = Path(r"C:\Users\thiago\Documents\estudos_python\etes_nesa")
DIR_BASE = RAIZ / "base_dados"
DIR_PRODUTOS = RAIZ / "Efluentes_Sanitários" / "produtos"
FILE_INPUT = DIR_BASE / "ETEs_NorteEnergia_Processado_v1.xlsx"
DIR_PRODUTOS.mkdir(parents=True, exist_ok=True)

NESA_COLORS = {'primary': '#009CA7', 'text': '#404041', 'muted': '#9aa0a6'}
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Segoe UI', 'Calibri', 'Arial', 'DejaVu Sans'],
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

try:
    print("[*] Lendo a base processada para auditoria de QA/QC...")
    df = pd.read_excel(FILE_INPUT)
    col_ete = [c for c in df.columns if 'DESCRICAO' in c.upper() and 'PONTO' in c.upper()][0]
    df[col_ete] = df[col_ete].astype(str).str.upper().str.strip()
    df[col_ete] = df[col_ete].replace({'ETE 1': 'ETE 01', 'ETE 2': 'ETE 02'})

    colunas_metadados = df.columns[:4].tolist()
    parametros = [c for c in df.columns if c not in colunas_metadados
                  and not c.endswith('_IMPUTADO_LQ') and not c.endswith('_STATUS_VMP')]

    dados_qc = []
    print("[*] Contabilizando matriz de dados censurados...")
    for param in parametros:
        col_flag = param + "_IMPUTADO_LQ"
        has_flag = col_flag in df.columns
        for ete in df[col_ete].unique():
            df_ete = df[df[col_ete] == ete]
            serie_param = df_ete[param].dropna()
            n_total = len(serie_param)
            if n_total > 0:
                n_censurado = (df_ete.loc[serie_param.index, col_flag].fillna(False).sum()
                               if has_flag else 0)
                dados_qc.append({
                    'ETE': ete, 'Parâmetro': param.replace('_', ' '),
                    'Total de Amostras (N)': int(n_total),
                    'Dados Censurados (<LQ)': int(n_censurado),
                    '% Censurado': round((n_censurado / n_total) * 100, 1),
                })

    df_qc = pd.DataFrame(dados_qc)
    df_qc.to_excel(DIR_PRODUTOS / "Relatorio_QAQC_Dados_Censurados.xlsx", index=False)

    df_qc_filtrado = df_qc[df_qc['% Censurado'] > 0]
    if not df_qc_filtrado.empty:
        pivot_qc = df_qc_filtrado.pivot(index='Parâmetro', columns='ETE', values='% Censurado').fillna(0)
        pivot_qc = pivot_qc.loc[pivot_qc.mean(axis=1).sort_values(ascending=False).index]

        fig, ax = plt.subplots(figsize=(9, max(6, len(pivot_qc) * 0.42)))
        sns.heatmap(pivot_qc, annot=True, fmt='.0f', cmap='YlGnBu', vmin=0, vmax=100,
                    linewidths=1.2, linecolor='white',
                    annot_kws={'fontsize': 9, 'fontweight': 'bold'},
                    cbar_kws={'label': '% de amostras censuradas (<LQ)', 'shrink': 0.55}, ax=ax)
        ax.set_xlabel(''); ax.set_ylabel(''); ax.tick_params(length=0)
        plt.setp(ax.get_xticklabels(), fontweight='bold')
        ax.text(0.0, 1.06, "Matriz de Dados Censurados (<LQ) por ETE", transform=ax.transAxes,
                fontsize=14, fontweight='bold', color='#2b2b2b', va='bottom', ha='left')
        ax.text(0.0, 1.015,
                "Células mais escuras = maior fração de amostras abaixo do limite de quantificação (não detectado)",
                transform=ax.transAxes, fontsize=9.5, color=NESA_COLORS['muted'], va='bottom', ha='left')
        fig.savefig(DIR_PRODUTOS / "Heatmap_QAQC_Censurados.png")
        plt.close(fig)

    print(f"\n[+] SUCESSO! Produtos QA/QC gerados em: {DIR_PRODUTOS}")
    print("\n=== TOP 10 MAIORES INCIDÊNCIAS DE CENSURA (<LQ) ===")
    print(df_qc.sort_values(by='% Censurado', ascending=False).head(10).to_string(index=False))
except Exception as e:
    print(f"[!] Erro no processamento de QA/QC: {e}")


[*] Lendo a base processada para auditoria de QA/QC...
[*] Contabilizando matriz de dados censurados...

[+] SUCESSO! Produtos QA/QC gerados em: C:\Users\thiago\Documents\estudos_python\etes_nesa\Efluentes_Sanitários\produtos

=== TOP 10 MAIORES INCIDÊNCIAS DE CENSURA (<LQ) ===
         ETE                                Parâmetro  Total de Amostras (N)  Dados Censurados (<LQ)  % Censurado
      ETE 01                            ARSENIO TOTAL                     26                      26        100.0
      ETE PM                            ARSENIO TOTAL                     20                      20        100.0
ETE COMPACTA                         CROMO TRIVALENTE                     24                      24        100.0
      ETE PM                         CROMO TRIVALENTE                     20                      20        100.0
      ETE 01 DICLOROETENO SOMATORIA DE 1112CIS12TRANS                     26                      26        100.0
      ETE 02                         

In [3]:
# =============================================================================
# Autor:   Thiago Souza (@engthiago1979-blip)  |  v3 (06-06-2026)
# Uso:     Dossiê Visual Segregado por UHE. P95 SEGMENTADO (BM=ETE01+02;
#          PM=ETE PM+Compacta). Escala FIXA + subtítulo honesto.
# Nota:    VMPs marcados como "conferir na base do projeto".
# =============================================================================

import datetime
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore", message="invalid value encountered")
warnings.filterwarnings("ignore", message="Mean of empty slice")

RAIZ = Path(r"C:\Users\thiago\Documents\estudos_python\etes_nesa")
DIR_BASE = RAIZ / "base_dados"
DIR_PRODUTOS = RAIZ / "Efluentes_Sanitários" / "produtos"
FILE_VAZAO = DIR_BASE / "vazão turbinada.xlsx"
FILE_INPUT = DIR_BASE / "ETEs_NorteEnergia_Processado_v1.xlsx"
DIR_PRODUTOS.mkdir(parents=True, exist_ok=True)

NESA_COLORS = {
    'rio': '#B9F8FF', 'ete_barras': '#009CA7', 'linha_vmp': '#F25A3C',
    'dbo': '#009CA7', 'nh3': '#52BD8B', 'fosforo': '#FEBD2A', 'lab': '#404041', 'muted': '#9aa0a6',
}
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Segoe UI', 'Calibri', 'Arial', 'DejaVu Sans'],
    'axes.axisbelow': True, 'savefig.dpi': 600, 'savefig.bbox': 'tight',
})

MESES_PT = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

def formatar_meses_pt(ax):
    def _fmt(x, pos=None):
        d = mdates.num2date(x)
        return f"{MESES_PT[d.month - 1]}/{d:%y}"
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(_fmt))

def titulo_storytelling(ax, titulo, subtitulo=None, fonte=None):
    ax.set_title('')
    ax.text(0.0, 1.12, titulo, transform=ax.transAxes, fontsize=15,
            fontweight='bold', color='#2b2b2b', va='bottom', ha='left')
    if subtitulo:
        ax.text(0.0, 1.035, subtitulo, transform=ax.transAxes, fontsize=10.5,
                color=NESA_COLORS['muted'], va='bottom', ha='left')
    if fonte:
        ax.text(0.0, -0.32, fonte, transform=ax.transAxes, fontsize=8,
                color=NESA_COLORS['muted'], va='top', ha='left')

def limpar_eixos(ax):
    for lado in ['top', 'right']:
        ax.spines[lado].set_visible(False)
    ax.tick_params(length=0)

def extrair_data_segura(val):
    if pd.isna(val):
        return pd.NaT
    if isinstance(val, (pd.Timestamp, datetime.datetime)):
        return pd.Timestamp(year=val.year, month=val.month, day=1)
    s = str(val).lower()
    meses = ['jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 'ago', 'set', 'out', 'nov', 'dez']
    for i, mes in enumerate(meses):
        if mes in s:
            anos = re.findall(r'\d{2,4}', s)
            if anos:
                ano = int(anos[-1]); ano += 2000 if ano < 100 else 0
                return pd.Timestamp(year=ano, month=i + 1, day=1)
    return pd.NaT

def calcular_p95_segmento(df, col_ete, etes_segmento, candidatos, fallback):
    df_seg = df if col_ete is None else df[df[col_ete].isin(etes_segmento)]
    for nome in candidatos:
        if nome in df_seg.columns:
            serie = pd.to_numeric(df_seg[nome], errors='coerce').dropna()
            if len(serie) >= 3:
                return (round(float(np.percentile(serie, 95)), 2),
                        f"P95 de '{nome}' em {etes_segmento} (N={len(serie)})")
    return fallback, f"FALLBACK (sem dados p/ {candidatos} em {etes_segmento})"

PARAM_COLS = {
    'DBO':  ['DEMANDA_BIOQUIMICA_DE_OXIGENIO'],
    'NH3':  ['NITROGENIO_AMONIACAL_TOTAL', 'NITROGENIO_AMONIACAL', 'N_AMONIACAL', 'NITROGENIO_AMONIACAL_NH3'],
    'FOSF': ['FOSFORO_TOTAL', 'FOSFORO', 'P_TOTAL', 'FOSFORO_TOTAL_P'],
}
FALLBACKS = {'DBO': 88.78, 'NH3': 45.0, 'FOSF': 4.5}
SEGMENTOS = {'BM': ['ETE 01', 'ETE 02'], 'PM': ['ETE PM', 'ETE COMPACTA']}

dados_gri = {
    'ETE_01': [651.4, 635.7, 503.4, 675.4, 639.4, 569.0, 566.6, 547.0, 455.1, 679.9, 581.8, 551.6, 324.8, 169.6, 99.2, 155.2, 129.6, 110.4, 137.6, 120.0, 146.4, 160.0, 144.8, 92.0, 107.2, 111.2, 145.6],
    'ETE_02': [597.4, 525.1, 415.0, 535.8, 579.8, 469.4, 534.2, 438.6, 330.9, 581.3, 405.8, 387.2, 2414.4, 1060.8, 959.2, 1325.6, 1130.4, 640.8, 885.6, 580.0, 664.8, 688.0, 585.6, 648.8, 602.4, 502.4, 623.76],
    'ETE_PM': [87.0, 198.4, 206.1, 185.6, 215.0, 245.1, 294.4, 258.4, 291.0, 338.5, 308.6, 314.5, 281.6, 146.4, 133.76, 137.92, 142.56, 141.12, 126.72, 104.16, 104.48, 0.0, 109.92, 0.0, 0.0, 57.6, 65.92],
    'ETE_COMPACTA': [21.8, 49.6, 51.5, 46.4, 50.6, 52.5, 52.8, 52.0, 45.4, 52.3, 46.2, 46.7, 41.6, 16.0, 18.24, 22.08, 23.04, 20.48, 18.88, 15.84, 19.76, 25.12, 19.68, 19.04, 14.72, 14.4, 16.48],
}
df_gri = pd.DataFrame(dados_gri)
df_gri['DATA_DT'] = pd.date_range(start='2024-01-01', periods=len(df_gri), freq='MS')
LIMITE_INICIO = pd.Timestamp('2023-12-15')
LIMITE_FIM = pd.Timestamp('2026-03-15')

try:
    p95, origem = {}, {}
    if FILE_INPUT.exists():
        df_proc = pd.read_excel(FILE_INPUT)
        col_ete = next((c for c in df_proc.columns if 'DESCRICAO' in c.upper() and 'PONTO' in c.upper()), None)
        if col_ete is not None:
            df_proc[col_ete] = (df_proc[col_ete].astype(str).str.upper().str.strip()
                                .replace({'ETE 1': 'ETE 01', 'ETE 2': 'ETE 02'}))
        for p, cand in PARAM_COLS.items():
            for seg, etes in SEGMENTOS.items():
                p95[(p, seg)], origem[(p, seg)] = calcular_p95_segmento(df_proc, col_ete, etes, cand, FALLBACKS[p])
    else:
        for p in PARAM_COLS:
            for seg in SEGMENTOS:
                p95[(p, seg)], origem[(p, seg)] = FALLBACKS[p], "FALLBACK (base não encontrada)"

    print("[*] P95 SEGMENTADO por UHE (rastreabilidade):")
    for seg in SEGMENTOS:
        print(f"  --- Segmento {seg} ({' + '.join(SEGMENTOS[seg])}) ---")
        for p in PARAM_COLS:
            print(f"      {p:5s} = {p95[(p, seg)]:8.2f} mg/L  ->  {origem[(p, seg)]}")

    df_vazao = pd.read_excel(FILE_VAZAO)
    df_vazao.columns = ['Mes_Original', 'Q_Pimental_m3s', 'Q_BeloMonte_m3s']
    df_vazao = df_vazao.dropna(subset=['Mes_Original'])
    df_vazao['DATA_DT'] = df_vazao['Mes_Original'].apply(extrair_data_segura)
    df_vazao = df_vazao.dropna(subset=['DATA_DT'])
    df_final = pd.merge(df_gri, df_vazao, on='DATA_DT', how='left')

    for seg, q_col, etes in [('BM', 'Q_BeloMonte_m3s', ['ETE_01', 'ETE_02']),
                             ('PM', 'Q_Pimental_m3s', ['ETE_PM', 'ETE_COMPACTA'])]:
        df_final[f'Vol_Rio_{seg}_m3mes'] = df_final[q_col] * 86400 * 30.44
        df_final[f'Vol_Efluente_{seg}'] = df_final[etes].sum(axis=1)
        df_final[f'Fator_Diluicao_{seg}'] = df_final[f'Vol_Rio_{seg}_m3mes'] / df_final[f'Vol_Efluente_{seg}']
        for nome in ['DBO', 'NH3', 'FOSF']:
            df_final[f'Adicao_{nome}_{seg}_mgL'] = (df_final[f'Vol_Efluente_{seg}'] * p95[(nome, seg)]) / df_final[f'Vol_Rio_{seg}_m3mes']

    print("\n" + "=" * 80)
    print("  DOSSIÊ VISUAL SEGREGADO POR UHE (v3 - P95 segmentado - Jan/24 a Mar/26)")
    print("=" * 80)

    def gerar_grafico_diluicao(fator_col, uhe_nome, arquivo_nome):
        fig, ax = plt.subplots(figsize=(14, 6)); limpar_eixos(ax)
        fator_milhoes = df_final[fator_col] / 1e6
        idx_min = fator_milhoes.idxmin() if fator_milhoes.notna().any() else None
        cores = [NESA_COLORS['linha_vmp'] if i == idx_min else NESA_COLORS['ete_barras'] for i in fator_milhoes.index]
        bars = ax.bar(df_final['DATA_DT'], fator_milhoes, width=22, color=cores, alpha=0.92)
        ax.bar_label(bars, labels=[f'{v:.1f}' if pd.notna(v) else '' for v in fator_milhoes],
                     padding=4, fontsize=8.5, color='#404041')
        ax.set_ylim(0, fator_milhoes.max() * 1.18 if pd.notna(fator_milhoes.max()) else 10)
        ax.set_xlim(LIMITE_INICIO, LIMITE_FIM)
        ax.grid(axis='y', linestyle='--', alpha=0.35); ax.grid(axis='x', visible=False)
        min_fator = fator_milhoes.min(); subt = ""
        if pd.notna(min_fator):
            x_min = df_final['DATA_DT'].iloc[df_final.index.get_loc(idx_min)]
            ax.annotate(f'Pior mês: 1 p/ {min_fator:.1f} M', xy=(x_min, min_fator), xytext=(0, 28),
                        textcoords='offset points', ha='center', fontsize=9.5, fontweight='bold',
                        color=NESA_COLORS['linha_vmp'], arrowprops=dict(arrowstyle='->', color=NESA_COLORS['linha_vmp'], lw=1.6))
            subt = f"Mesmo no pior mês histórico, cada 1 L de efluente é diluído em {min_fator:.1f} milhões de litros do Rio Xingu"
        titulo_storytelling(ax, f"Proporção Hidrodinâmica - UHE {uhe_nome}", subt,
                            "Fonte: Volumes GRI 303-4 (NESA) x Vazão turbinada (ONS)   |   Vol.rio = Q x 86400 x 30,44")
        ax.set_ylabel("Milhões de litros de rio por 1 L de efluente", fontweight='bold')
        formatar_meses_pt(ax); plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        fig.savefig(DIR_PRODUTOS / arquivo_nome); plt.close(fig)

    def gerar_grafico_impacto(param_col, cor, nome_parametro, p95_seg, limite_lab, nota_conama, limite_y, uhe_nome, arquivo_nome):
        fig, ax = plt.subplots(figsize=(12, 5.4)); limpar_eixos(ax)
        mask = df_final[param_col].notna()
        ax.plot(df_final['DATA_DT'], df_final[param_col], color=cor, linewidth=3, label=f'Adição efetiva de {nome_parametro}', zorder=3)
        ax.fill_between(df_final['DATA_DT'][mask], 0, df_final[param_col][mask], color=cor, alpha=0.18)
        ax.axhline(limite_lab, color=NESA_COLORS['lab'], linestyle='-', linewidth=2, label=f'Limite de detecção lab. ({limite_lab} mg/L)')
        ax.set_ylim(0, limite_y); ax.set_xlim(LIMITE_INICIO, LIMITE_FIM)
        ax.annotate(nota_conama, xy=(pd.Timestamp('2025-01-01'), limite_y * 0.96),
                    xytext=(pd.Timestamp('2025-01-01'), limite_y * 0.82), color=NESA_COLORS['linha_vmp'],
                    fontweight='bold', fontsize=10.5, ha='center',
                    arrowprops=dict(facecolor=NESA_COLORS['linha_vmp'], arrowstyle='->', lw=2))
        adic_max = df_final[param_col].max()
        if pd.notna(adic_max) and adic_max > 0:
            subt = (f"P95 do efluente = {p95_seg:g} mg/L -> adição máxima de {adic_max:.5f} mg/L  "
                    f"({limite_lab / adic_max:,.0f}x abaixo do limite de detecção do laboratório)")
        else:
            subt = f"P95 do efluente = {p95_seg:g} mg/L -> adição efetiva próxima de zero após diluição"
        titulo_storytelling(ax, f"Modelagem de Risco - UHE {uhe_nome}: {nome_parametro}", subt,
                            "Fonte: Balanço de massa NESA (P95 segmentado x diluição do Rio Xingu)   |   VMP CONAMA 430/11 - conferir na base do projeto")
        ax.set_ylabel("Concentração adicionada (mg/L)", fontweight='bold')
        ax.grid(axis='y', linestyle=':', alpha=0.5); ax.grid(axis='x', visible=False)
        formatar_meses_pt(ax); ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=2)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        fig.savefig(DIR_PRODUTOS / arquivo_nome); plt.close(fig)

    print("[*] Gerando gráficos para UHE Belo Monte...")
    gerar_grafico_diluicao('Fator_Diluicao_BM', 'Belo Monte', "Produto1A_Diluicao_BeloMonte.png")
    gerar_grafico_impacto('Adicao_DBO_BM_mgL', NESA_COLORS['dbo'], 'DBO', p95[('DBO', 'BM')], 2.0,
                          'Limite CONAMA 120 mg/L (conferir na base) -> Fora de escala', 2.5, 'Belo Monte', "Produto2A_Impacto_DBO_BM.png")
    gerar_grafico_impacto('Adicao_NH3_BM_mgL', NESA_COLORS['nh3'], 'N. Amoniacal', p95[('NH3', 'BM')], 0.5,
                          'Limite CONAMA 20 mg/L (conferir na base) -> Fora de escala\n*Isento pelo Art. 21', 0.7, 'Belo Monte', "Produto3A_Impacto_NH3_BM.png")
    gerar_grafico_impacto('Adicao_FOSF_BM_mgL', NESA_COLORS['fosforo'], 'Fósforo Total', p95[('FOSF', 'BM')], 0.5,
                          'Risco de eutrofização -> Fora de escala', 0.7, 'Belo Monte', "Produto4A_Impacto_Fosforo_BM.png")

    print("[*] Gerando gráficos para UHE Pimental...")
    gerar_grafico_diluicao('Fator_Diluicao_PM', 'Pimental (TVR)', "Produto1B_Diluicao_Pimental.png")
    gerar_grafico_impacto('Adicao_DBO_PM_mgL', NESA_COLORS['dbo'], 'DBO', p95[('DBO', 'PM')], 2.0,
                          'Limite CONAMA 120 mg/L (conferir na base) -> Fora de escala', 2.5, 'Pimental', "Produto2B_Impacto_DBO_PM.png")
    gerar_grafico_impacto('Adicao_NH3_PM_mgL', NESA_COLORS['nh3'], 'N. Amoniacal', p95[('NH3', 'PM')], 0.5,
                          'Limite CONAMA 20 mg/L (conferir na base) -> Fora de escala\n*Isento pelo Art. 21', 0.7, 'Pimental', "Produto3B_Impacto_NH3_PM.png")
    gerar_grafico_impacto('Adicao_FOSF_PM_mgL', NESA_COLORS['fosforo'], 'Fósforo Total', p95[('FOSF', 'PM')], 0.5,
                          'Risco de eutrofização -> Fora de escala', 0.7, 'Pimental', "Produto4B_Impacto_Fosforo_PM.png")

    df_final['Mes/Ano'] = df_final['DATA_DT'].dt.strftime('%m/%Y')
    df_final['P95_DBO_BM'] = p95[('DBO', 'BM')]; df_final['P95_DBO_PM'] = p95[('DBO', 'PM')]
    colunas_export = ['Mes/Ano', 'Q_BeloMonte_m3s', 'Vol_Rio_BM_m3mes', 'Vol_Efluente_BM', 'Fator_Diluicao_BM',
                      'Adicao_DBO_BM_mgL', 'Adicao_NH3_BM_mgL', 'Adicao_FOSF_BM_mgL',
                      'Q_Pimental_m3s', 'Vol_Rio_PM_m3mes', 'Vol_Efluente_PM', 'Fator_Diluicao_PM',
                      'Adicao_DBO_PM_mgL', 'Adicao_NH3_PM_mgL', 'Adicao_FOSF_PM_mgL']
    df_final[colunas_export].to_excel(DIR_PRODUTOS / "Relatorio_Diluicao_Mestre_Segregado.xlsx", index=False)
    print("\n[*] Concluído! 8 gráficos segregados (P95 por UHE) salvos na pasta 'produtos'.")
except Exception as e:
    print(f"\n[!] ERRO CRÍTICO ENCONTRADO: {e}")


[*] P95 SEGMENTADO por UHE (rastreabilidade):
  --- Segmento BM (ETE 01 + ETE 02) ---
      DBO   =    86.12 mg/L  ->  P95 de 'DEMANDA_BIOQUIMICA_DE_OXIGENIO' em ['ETE 01', 'ETE 02'] (N=78)
      NH3   =   266.05 mg/L  ->  P95 de 'NITROGENIO_AMONIACAL' em ['ETE 01', 'ETE 02'] (N=78)
      FOSF  =     8.80 mg/L  ->  P95 de 'FOSFORO_TOTAL' em ['ETE 01', 'ETE 02'] (N=78)
  --- Segmento PM (ETE PM + ETE COMPACTA) ---
      DBO   =   188.37 mg/L  ->  P95 de 'DEMANDA_BIOQUIMICA_DE_OXIGENIO' em ['ETE PM', 'ETE COMPACTA'] (N=85)
      NH3   =   184.16 mg/L  ->  P95 de 'NITROGENIO_AMONIACAL' em ['ETE PM', 'ETE COMPACTA'] (N=85)
      FOSF  =    27.83 mg/L  ->  P95 de 'FOSFORO_TOTAL' em ['ETE PM', 'ETE COMPACTA'] (N=85)

  DOSSIÊ VISUAL SEGREGADO POR UHE (v3 - P95 segmentado - Jan/24 a Mar/26)
[*] Gerando gráficos para UHE Belo Monte...
[*] Gerando gráficos para UHE Pimental...

[*] Concluído! 8 gráficos segregados (P95 por UHE) salvos na pasta 'produtos'.
